# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://doi.org/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library following Croissant schema standards.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# The mlcroissant 'metadata' is a custom object; use to_json() to serialize as a dictionary easily.
metadata_json = dataset.metadata.to_json()
print(f"Dataset: {metadata_json['name']}\nDescription: {metadata_json['description']}")

## 2. Data Overview
Review available record sets and their fields using their `@id` attributes (as required by the Croissant schema standard and `mlcroissant`).

In [ ]:
# List all record sets
record_sets = dataset.record_sets
print(f"Record sets found: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record Set @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    # List fields in this record set
    print("  Fields:")
    for field in rs.fields:
        print(f"    Field @id: {field.id} | Name: {field.name} | Data type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from all main record sets into DataFrames for analysis. Use the record set `@id` and the fields identified previously.

In [ ]:
# Build a dictionary of DataFrames for each record set
dataframes = {}

for rs in record_sets:
    rs_id = rs.id
    # Load records for this record set (each record is a dict mapping field @id to value)
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} rows for record set: {rs_id}")

# For demonstration, pick the main survey record set (assumed to be the largest)
if len(dataframes) > 0:
    main_rs_id = max(dataframes, key=lambda k: dataframes[k].shape[0])
    print(f"\nMain record set selected for analysis: {main_rs_id}")
    print("Columns in main record set (field @id):")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No dataframes could be constructed from the available record sets.")

## 4. Exploratory Data Analysis (EDA)
Demonstrate basic data processing: filtering, normalization, and grouping. All field and record set references use their Croissant `@id`s as required.

In [ ]:
import numpy as np

# Pick a known numeric field (use GAD-7 score if available)
# Use field @id (often like 'gad7_total' or similar in datasets, confirm with printout above)
main_df = dataframes[main_rs_id]
numeric_candidates = [col for col in main_df.columns if main_df[col].dtype in [np.float64, np.int64]]
if not numeric_candidates:
    # Try to infer from column names containing typical survey score variables
    for probe in ['gad7', 'phq9', 'psq', 'score']:
        found = [col for col in main_df.columns if probe in col.lower()]
        if found:
            numeric_candidates.extend(found)

if numeric_candidates:
    numeric_field = numeric_candidates[0]  # Choose first candidate
    print(f"Using numeric field for analysis: {numeric_field}")
else:
    # Fallback to any column
    numeric_field = main_df.columns[0]
    print(f"No obvious numeric field found, using: {numeric_field}")

# Set a threshold (arbitrary/illustrative, e.g., >10)
threshold = 10
if pd.api.types.is_numeric_dtype(main_df[numeric_field]):
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold} (n={len(filtered_df)}):")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field (e.g. gender, look for field with few unique values <20 and type is object)
    group_field_candidates = [col for col in main_df.columns if (main_df[col].nunique()<20 and main_df[col].dtype==object)]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        grouped_df = (
            filtered_df.groupby(group_field)[numeric_field].mean().reset_index().rename(columns={numeric_field: f"mean_{numeric_field}"})
        )
        print(f"Grouped data by {group_field} (@id):")
        display(grouped_df)
    else:
        print("No suitable group field found.")
else:
    print(f"{numeric_field} could not be interpreted as numeric.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset (using `matplotlib` or `seaborn`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=20)
plt.xlabel(numeric_field)
plt.title(f"Distribution of {numeric_field} scores")
plt.show()

# If group_field was found, show boxplot comparison
if 'group_field' in locals():
    plt.figure(figsize=(8,5))
    sns.boxplot(data=main_df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
This notebook demonstrated loading and processing the Kilifi Mental Health Survey dataset using the `mlcroissant` library. We explored dataset structure, extracted record sets by their `@id`, performed EDA and simple visualizations. For further research, refine data selection using field definitions from the Croissant schema and apply advanced statistical or ML methods as needed.